# FlipItNews — End-to-End News Classification
This notebook covers EDA, preprocessing, BoW/TF-IDF/Word2Vec features, NN/RNN models, evaluation, and saving the RNN artifacts for Streamlit.

**Dataset note:** The supplied file is tab-separated despite its `.csv` extension. It contains `category`, `filename`, `title`, and `content`.

In [ ]:
import os, re, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

DATA_PATH = Path("data/bbc-news-data.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/bbc-news-data.csv")
df = pd.read_csv(DATA_PATH, sep="\t")
print("Shape:", df.shape)
display(df.head())
display(df.info())

## 1. EDA

In [ ]:
print("Columns:", df.columns.tolist())
print("\\nMissing values:\\n", df.isna().sum())
print("\\nDuplicate rows:", df.duplicated().sum())
print("\\nCategory counts:\\n", df["category"].value_counts())

plt.figure(figsize=(9, 4))
sns.countplot(data=df, x="category", order=df["category"].value_counts().index)
plt.title("News articles by category")
plt.xlabel("Category"); plt.ylabel("Number of articles")
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

df["text_length"] = df["content"].fillna("").str.split().str.len()
plt.figure(figsize=(9, 4))
sns.histplot(data=df, x="text_length", hue="category", bins=40, element="step", stat="density", common_norm=False)
plt.title("Article length distribution by category")
plt.tight_layout(); plt.show()

display(df.groupby("category")["text_length"].describe().round(1))

## 2. Cleaning and preprocessing
We combine title and content because both can contain useful classification signals. Stop-word removal is used for the BoW/TF-IDF/Word2Vec experiments. For the RNN, we retain a sequence-friendly version with punctuation cleaned; removing every stop word can remove useful context.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

df = df.drop_duplicates().copy()
df["category"] = df["category"].astype(str).str.strip()
df["title"] = df["title"].fillna("").astype(str)
df["content"] = df["content"].fillna("").astype(str)
df["text"] = (df["title"] + " " + df["content"]).str.strip()

def clean_text(text, remove_stopwords=True):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    tokens = re.findall(r"[a-z0-9]+", text)
    if remove_stopwords:
        tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS]
    return " ".join(tokens)

df["clean_text"] = df["text"].apply(lambda x: clean_text(x, True))
df["rnn_text"] = df["text"].apply(lambda x: clean_text(x, False))
display(df[["category", "title", "clean_text"]].head(3))

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["category"])
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"], y, test_size=0.2, random_state=SEED, stratify=y
)
print("Classes:", list(label_encoder.classes_))
print("Train:", len(X_train_text), "Test:", len(X_test_text))

## 3. BoW and TF-IDF + Neural Network
The vectorizers are fitted on training text only to prevent test-data leakage.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.pipeline import make_pipeline

results = []
fitted_models = {}

def evaluate_model(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    row = {
        "Model": name,
        "Accuracy": accuracy_score(yte, pred),
        "Precision_macro": precision_score(yte, pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(yte, pred, average="macro", zero_division=0),
        "F1_macro": f1_score(yte, pred, average="macro", zero_division=0),
    }
    results.append(row)
    fitted_models[name] = model
    print(name)
    print(classification_report(yte, pred, target_names=label_encoder.classes_, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(yte, pred, display_labels=label_encoder.classes_, xticks_rotation=45, cmap="Blues")
    plt.tight_layout(); plt.show()
    return row

bow = CountVectorizer(max_features=20000, ngram_range=(1,2), min_df=2)
Xtr_bow = bow.fit_transform(X_train_text)
Xte_bow = bow.transform(X_test_text)
bow_nn = MLPClassifier(hidden_layer_sizes=(128,), activation="relu", max_iter=25, early_stopping=True, random_state=SEED)
evaluate_model("BoW + MLP (NN)", bow_nn, Xtr_bow, Xte_bow, y_train, y_test)

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=2, sublinear_tf=True)
Xtr_tfidf = tfidf.fit_transform(X_train_text)
Xte_tfidf = tfidf.transform(X_test_text)
tfidf_nn = MLPClassifier(hidden_layer_sizes=(128,), activation="relu", max_iter=25, early_stopping=True, random_state=SEED)
evaluate_model("TF-IDF + MLP (NN)", tfidf_nn, Xtr_tfidf, Xte_tfidf, y_train, y_test)

## 4. Word2Vec + NN
Word2Vec is trained on training articles only. We average token vectors to create one fixed-length document vector for a fair feed-forward baseline.

In [ ]:
from gensim.models import Word2Vec

train_tokens = [s.split() for s in X_train_text]
w2v = Word2Vec(sentences=train_tokens, vector_size=100, window=5, min_count=2, workers=1, seed=SEED, epochs=20)

def average_vectors(texts, model, dim=100):
    out = np.zeros((len(texts), dim), dtype=np.float32)
    for i, text in enumerate(texts):
        vecs = [model.wv[t] for t in text.split() if t in model.wv]
        if vecs:
            out[i] = np.mean(vecs, axis=0)
    return out

Xtr_w2v = average_vectors(X_train_text, w2v)
Xte_w2v = average_vectors(X_test_text, w2v)
w2v_nn = MLPClassifier(hidden_layer_sizes=(128,), max_iter=100, early_stopping=True, random_state=SEED)
evaluate_model("Word2Vec mean + MLP (NN)", w2v_nn, Xtr_w2v, Xte_w2v, y_train, y_test)

## 5. RNN/LSTM model
The Keras Tokenizer and padding length are saved with the model. This exact tokenizer must be reused at inference time.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models, callbacks

tf.keras.utils.set_random_seed(SEED)
# Use cleaned text without stop-word removal for the sequence model.
rnn_train, rnn_test, y_rnn_train, y_rnn_test = train_test_split(
    df["rnn_text"], y, test_size=0.2, random_state=SEED, stratify=y
)
rnn_train, rnn_val, y_rnn_train, y_rnn_val = train_test_split(
    rnn_train, y_rnn_train, test_size=0.15, random_state=SEED, stratify=y_rnn_train
)
MAX_WORDS = 20000
MAX_LEN = 400
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(rnn_train)
Xtr_seq = pad_sequences(tokenizer.texts_to_sequences(rnn_train), maxlen=MAX_LEN, padding="post", truncating="post")
Xval_seq = pad_sequences(tokenizer.texts_to_sequences(rnn_val), maxlen=MAX_LEN, padding="post", truncating="post")
Xtest_seq = pad_sequences(tokenizer.texts_to_sequences(rnn_test), maxlen=MAX_LEN, padding="post", truncating="post")

rnn = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(MAX_WORDS, 128, mask_zero=True),
    layers.Bidirectional(layers.LSTM(64, dropout=0.2)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(label_encoder.classes_), activation="softmax")
])
rnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
rnn.summary()

early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history = rnn.fit(
    Xtr_seq, y_rnn_train,
    validation_data=(Xval_seq, y_rnn_val),
    epochs=12, batch_size=32,
    callbacks=[early_stop], verbose=1
)

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.title("RNN training history")
plt.tight_layout(); plt.show()

rnn_prob = rnn.predict(Xtest_seq, verbose=0)
rnn_pred = rnn_prob.argmax(axis=1)
results.append({
    "Model": "BiLSTM RNN",
    "Accuracy": accuracy_score(y_rnn_test, rnn_pred),
    "Precision_macro": precision_score(y_rnn_test, rnn_pred, average="macro", zero_division=0),
    "Recall_macro": recall_score(y_rnn_test, rnn_pred, average="macro", zero_division=0),
    "F1_macro": f1_score(y_rnn_test, rnn_pred, average="macro", zero_division=0),
})
print(classification_report(y_rnn_test, rnn_pred, target_names=label_encoder.classes_, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_rnn_test, rnn_pred, display_labels=label_encoder.classes_, xticks_rotation=45, cmap="Blues")
plt.tight_layout(); plt.show()

## 6. Compare models

In [ ]:
results_df = pd.DataFrame(results).sort_values("F1_macro", ascending=False).reset_index(drop=True)
display(results_df.style.format({c: "{:.4f}" for c in results_df.columns if c != "Model"}))
ax = results_df.set_index("Model")[["Accuracy", "Precision_macro", "Recall_macro", "F1_macro"]].plot(kind="bar", figsize=(12,5))
ax.set_ylim(0, 1); ax.set_ylabel("Score"); ax.set_title("News classification model comparison")
plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.show()

## 7. Save deployment artifacts

In [ ]:
from pathlib import Path
import json
ARTIFACT_DIR = Path("models")
ARTIFACT_DIR.mkdir(exist_ok=True)
rnn.save(ARTIFACT_DIR / "best_rnn.keras")
with open(ARTIFACT_DIR / "tokenizer.json", "w", encoding="utf-8") as f:
    f.write(tokenizer.to_json())
with open(ARTIFACT_DIR / "label_classes.json", "w", encoding="utf-8") as f:
    json.dump(label_encoder.classes_.tolist(), f)
with open(ARTIFACT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump({"max_words": MAX_WORDS, "max_len": MAX_LEN}, f)
results_df.to_csv(ARTIFACT_DIR / "model_comparison.csv", index=False)
print("Saved model and preprocessing artifacts in:", ARTIFACT_DIR.resolve())

## 8. Business interpretation
- **Content organization:** automatically suggest a category when an article is submitted.
- **Search:** use predicted labels to filter and retrieve articles by topic.
- **Personalization:** match article categories to a reader's chosen interests; production personalization would need user preferences and engagement data.
- **Human review:** low-confidence or ambiguous predictions should be reviewed by an editor.
- **Monitoring:** track category-level F1 and changes in article topics after deployment.